In [3]:
import os, sys
# os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
os.chdir("..")
sys.path.append('.')

from collections import defaultdict
import regex as re
from tqdm import tqdm

In [4]:
# pattern検出
def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)

In [5]:
def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)
    
    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts

In [6]:
def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

In [13]:
def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 特殊トークンで分割
    texts = input_text.split(end_token)

    # 各テキスト片を事前トークン化
    pretoken_counts = defaultdict(int)
    for text in tqdm(texts, desc="Pretokeing"):
        for pretoken in pretokenize(text):
            pretoken_counts[pretoken] += 1
    
    # 事前トークンをID列に変換
    ids_counts = {
        tuple(pretoken.encode("utf-8")): 
        count for pretoken, count in pretoken_counts.items()
    }

    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):
        # 各ids列について、その出現回数を考慮してペア頻度を集計
        pair_counts = defaultdict(int)
        for ids, count in ids_counts.items():
            count_pairs(ids, count, pair_counts)
        
        # ペアが存在しない倍の処理
        if not pair_counts:
            break

        # 最頻出ペアを選択
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 各ids列をマージして更新
        new_id_counts = defaultdict(int)
        for ids, count in ids_counts.items():
            new_ids = merge(ids, best_pair, new_id)
            new_id_counts[tuple(new_ids)] += count
        ids_counts = new_id_counts
    
    return merge_rules

In [14]:
vocab_size = 1000  # 語彙サイズの設定
file_path = "codebot/tiny_codes.txt"
text = open(file_path).read()
merge_rules = train_bpe(text, vocab_size)

Training BPE: 100%|██████████| 743/743 [00:21<00:00, 34.08it/s]


In [15]:
print(merge_rules)

{(32, 32): 256, (256, 256): 257, (105, 110): 258, (256, 32): 259, (114, 101): 260, (111, 114): 261, (32, 61): 262, (115, 116): 263, (116, 101): 264, (10, 259): 265, (115, 101): 266, (100, 101): 267, (32, 116): 268, (111, 110): 269, (108, 101): 270, (114, 97): 271, (10, 257): 272, (32, 97): 273, (32, 105): 274, (104, 101): 275, (32, 260): 276, (97, 116): 277, (109, 101): 278, (114, 258): 279, (32, 99): 280, (32, 102): 281, (97, 114): 282, (117, 114): 283, (101, 114): 284, (41, 58): 285, (32, 112): 286, (109, 112): 287, (108, 105): 288, (103, 101): 289, (32, 258): 290, (32, 110): 291, (117, 109): 292, (272, 259): 293, (116, 105): 294, (268, 275): 295, (97, 108): 296, (110, 100): 297, (114, 111): 298, (32, 115): 299, (261, 116): 300, (283, 110): 301, (110, 116): 302, (116, 301): 303, (267, 102): 304, (32, 39): 305, (279, 116): 306, (276, 303): 307, (32, 101): 308, (287, 300): 309, (117, 116): 310, (99, 101): 311, (117, 101): 312, (32, 109): 313, (32, 49): 314, (108, 97): 315, (115, 115): 